# laya-idjvsuen v4 — Fine-tune *general* dari v1 (Google Colab, T4)

Fine-tune **[faall7479/laya-idjvsuen-v1](https://huggingface.co/faall7479/laya-idjvsuen-v1)** ke arah **umum/general** (bukan khusus ticketing) dengan data baru:
- **SIB-200** — klasifikasi topik 7 kategori, paralel `id`/`jv`/`sun`/`en` (manusia-verifiable, FLORES-200)
- **IndoNLU** (opsional otomatis) — emosi, sentimen app-review, aspek (`id`)
- **Code-switch sintetis** — pasangan SIB-200 se-kategori (eval)
- **Replay data v1** — MASSIVE + NusaX + CS (43k item) agar kemampuan lama tidak dilupakan

**Prasyarat**
1. Runtime GPU: *Runtime → Change runtime type → T4 GPU* (16 GB; fp16 — T4 tidak punya bf16).
2. Upload **sekali** `train_items.pt` (76 MB, dari `D:\MyTools\laya\data\processed\`) ke folder **`laya/`** di Google Drive Anda.
3. Opsional: token HF (write) untuk mem-publish hasil.

Estimasi: dataset ~47k item × 2 epoch ≈ **2,5–4,5 jam** di T4. Checkpoint per epoch tersimpan ke Drive (`--resume` aman dari disconnect).

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Aktifkan GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# Setup: dependensi + repo laya (upstream SDK) + repo pipeline ini
!pip -q install "datasets<3.0" transformers safetensors bitsandbytes
!git clone -q https://github.com/NandhaKishorM/laya /content/laya-repo
!git clone -q https://github.com/muhfalihr/laya-idjvsuen /content/repo
%env PYTHONPATH=/content/laya-repo
import os
os.chdir("/content/repo")
print("siap:", os.getcwd())

In [ ]:
# Mount Drive + salin data replay v1 (train_items.pt harus sudah di Drive/laya/)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
os.makedirs("data/processed", exist_ok=True)
for name in ("train_items.pt", "test_sets.pt"):
    p = f"/content/drive/MyDrive/laya/{name}"
    if os.path.exists(p):
        shutil.copy(p, f"data/processed/{name}")
        print("siap:", name)
    else:
        print("TIDAK ADA:", p, "→ akan jalan tanpa replay (kurang disarankan)")

In [ ]:
# Unduh base model v1 dari Hugging Face
from huggingface_hub import snapshot_download
BASE = snapshot_download("faall7479/laya-idjvsuen-v1", local_dir="/content/base-v1")
print("base:", BASE)

In [ ]:
# Bangun dataset general v4 (SIB-200 + IndoNLU + CS + replay)
!python scripts/15_build_general.py --model /content/base-v1 --replay data/processed/train_items.pt

## Training
Catatan penting untuk T4: **`--amp fp16`** (T4/Turing tidak mendukung bf16; GradScaler diaktifkan otomatis oleh skrip), `--optim adamw8bit` menghemat VRAM. Output langsung ke Drive supaya selamat dari disconnect.

In [ ]:
OUT = "/content/drive/MyDrive/laya/runs/laya-idjvsuen-v4"
!python scripts/04_train.py --model /content/base-v1 \
    --data data/processed/train_items_general.pt \
    --out "{OUT}" \
    --model-name laya-idjvsuen-v4-general \
    --amp fp16 --optim adamw8bit \
    --micro-batch 8 --grad-accum 8 --epochs 2

### Kalau sesi terputus di tengah jalan
Ulangi sel Setup, sel Drive, sel unduh base, dan sel bangun dataset di atas, lalu jalankan sel ini (perhatikan `--resume` — lanjut dari `checkpoint_latest` terakhir di Drive).

In [ ]:
OUT = "/content/drive/MyDrive/laya/runs/laya-idjvsuen-v4"
!python scripts/04_train.py --model /content/base-v1 \
    --data data/processed/train_items_general.pt \
    --out "{OUT}" --model-name laya-idjvsuen-v4-general \
    --amp fp16 --optim adamw8bit \
    --micro-batch 8 --grad-accum 8 --epochs 2 --resume

In [ ]:
# Evaluasi: set BARU (sib_*, cs_sib_*, emot/smsa/casa) + set LAMA (intent_*, sentiment_*, cs_*)
# -> angka lama harus stabil (efek replay), angka baru = kemampuan general yang ditambahkan
!python scripts/05_eval.py --model "{OUT}" --test-sets data/processed/test_sets_v4.pt \
    --amp fp16 --tag v4-general

## Publish ke Hugging Face (opsional)
Model ini **general** — bukan pengganti `laya-idjvsuen-v3` untuk routing tiket. Ingat konvensi repo: setiap versi model baru → update tabel versi di README + docs.

In [ ]:
# from huggingface_hub import login; login()  # token dengan akses write
# !python scripts/07_publish_hf.py --repo-id faall7479/laya-idjvsuen-v4 --model-dir "{OUT}"